# API 综合交付实践

学习目标：逐步组合一个学习记录 API，核对配置、数据库、身份与归属检查的配合，并用固定请求流程检查接口契约和资源清理。

前置知识：FastAPI 路由与依赖注入、Pydantic 模型、SQL 事务、身份认证与授权、上下文管理器、接口测试。

适用版本：Python 3.12、FastAPI 0.141.1、Pydantic 2、SQLModel 0.0.42、SQLAlchemy 2.0。

环境准备：[FastAPI 环境与运行入口](README.md)。

工作目录：content/Web与应用开发/FastAPI。运行入口是本篇 Notebook：从空内核按顺序执行，末节删除本次临时数据库。中途停止时，先让当前 TestClient 的 with 退出，再执行末节清理。

示例全部使用应用内请求，不监听真实端口。两个用户的临时 Bearer 凭据只保存在运行时内存中，不输出、不写文件；这里不实现账号登录。

## 1 只配置本实验使用的数据库路径

本例管理标题形式的学习记录：每条记录属于一个用户，用户只能读取和修改自己的记录。同一用户不能使用重复标题；不同用户可以使用相同标题。

BaseSettings 把配置集中成有类型的对象。database_path 可以由 DELIVERY_DATABASE_PATH 环境变量提供，也可以在构造 Settings 时显式传入。实验采用后一种方式，保证每次使用新建的临时目录，不修改终端环境。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="DELIVERY_")
    database_path: Path


work_directory = TemporaryDirectory(prefix="fastapi-delivery-")
settings = Settings(database_path=Path(work_directory.name) / "records.sqlite3")
print("数据库文件名：", settings.database_path.name)  # 预期：数据库文件名： records.sqlite3。
# 这里只准备路径，连接数据库和建表放在应用启动时执行。
assert not settings.database_path.exists()

数据库文件名： records.sqlite3


## 2 区分存储字段、输入字段和输出字段

数据库表保存 id、title 和 owner_id。UniqueConstraint 的两个字段组成联合唯一约束：不允许两行同时具有相同的 owner_id 和 title。把约束放在数据库中，提交写入时仍会检查它。

SQLModel 的 table=True 表示表模型；id 尚未写入时可以为 None，插入后由数据库生成整数编号。这里只定义表，还没有创建数据库连接。

In [2]:
from sqlalchemy import UniqueConstraint
from sqlmodel import Field, Session, SQLModel, create_engine, select


class Record(SQLModel, table=True):
    __tablename__ = "delivery_record"
    __table_args__ = (UniqueConstraint("owner_id", "title"),)

    id: int | None = Field(default=None, primary_key=True)
    title: str
    owner_id: str


print("数据库字段：", list(Record.model_fields))  # 预期：数据库字段： ['id', 'title', 'owner_id']。

数据库字段： ['id', 'title', 'owner_id']


RecordInput 只接收标题，拒绝额外字段；创建和改名都使用这份输入。RecordOutput 只返回编号和标题，把 owner_id 留在服务端用于权限判断。from_attributes=True 允许从数据库对象的属性构造输出模型。

ErrorBody 描述本例主动返回的错误体。后面通过 responses 声明这些错误，用实际请求检查声明是否正确；声明本身不会产生错误响应。

In [3]:
from pydantic import BaseModel, ConfigDict, Field as ValidationField


class RecordInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str = ValidationField(min_length=1, max_length=80)


class RecordOutput(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    title: str


class ErrorBody(BaseModel):
    detail: str


errors = {
    code: {"model": ErrorBody} for code in (401, 403, 404, 409)
}
print("输入：", list(RecordInput.model_fields))  # 预期：输入： ['title']。
print("输出：", list(RecordOutput.model_fields))  # 预期：输出： ['id', 'title']。

输入： ['title']
输出： ['id', 'title']


## 3 在应用生命周期内管理 Engine

Engine 管理连接资源，随应用启动创建，随应用退出释放。lifespan 的 yield 前初始化，finally 中调用 dispose；app.state 保存本次运行的 Engine，退出后置回 None。

create_all 只用于本实验的新数据库建表，不会替已有数据执行字段迁移。check_same_thread=False 允许文件 SQLite 连接用于同步接口的线程切换，并不允许多个请求并发共用一个 Session。

In [4]:
from contextlib import asynccontextmanager

from fastapi import FastAPI

lifecycle_events = []


@asynccontextmanager
async def lifespan(app: FastAPI):
    engine = create_engine(
        f"sqlite:///{settings.database_path.as_posix()}",
        connect_args={"check_same_thread": False},
    )
    try:
        SQLModel.metadata.create_all(engine)
        app.state.engine = engine
        lifecycle_events.append("启动")
        yield
    finally:
        engine.dispose()
        app.state.engine = None
        lifecycle_events.append("释放")


app = FastAPI(title="学习记录 API", lifespan=lifespan)

使用 with TestClient(app) 才会运行这里的 lifespan。先做一次空表查询，观察启动和退出；之后各组请求会再次启动同一个应用，读取同一数据库文件里的已提交数据。

In [5]:
from fastapi.testclient import TestClient

with TestClient(app):
    with Session(app.state.engine) as session:
        assert session.exec(select(Record)).all() == []
    print("启动后数据库存在：", settings.database_path.is_file())  # 预期：启动后数据库存在： True。
assert app.state.engine is None
assert lifecycle_events == ["启动", "释放"]
print("生命周期：", lifecycle_events)  # 预期：生命周期： ['启动', '释放']。
# Engine 已释放，但数据库文件仍在；释放连接不等于删除持久化数据。
assert settings.database_path.is_file()

启动后数据库存在： True
生命周期： ['启动', '释放']


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 4 每次请求取得会话和当前用户

get_session 为一次请求提供一个 Session。with 在请求结束时关闭会话，归还连接资源；写入成功仍须在路由中显式提交，关闭会话不会代替提交。不要把同一个 Session 放进全局变量供并发请求共享。

SessionDep 是依赖的类型别名，后面的路由直接使用它。Request 用来访问当前应用的资源。

In [6]:
from typing import Annotated

from fastapi import Depends, HTTPException, Request, Response, Security


def get_session(request: Request):
    with Session(request.app.state.engine) as session:
        yield session


SessionDep = Annotated[Session, Depends(get_session)]

再为 Alice 和 Bob 生成两份临时凭据。secrets.token_urlsafe(32) 使用 32 个随机字节生成适合放进请求的字符串；这里不固定随机种子，也不显示字符串内容。

HTTPBearer 只负责读取 Bearer 凭据，get_user 再从本地映射取得用户标识。缺失或无效时返回 401 并给出 Bearer 挑战头。此映射是教学身份输入，不包含登录、到期或撤销流程。

In [7]:
from secrets import token_urlsafe

from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer

tokens = {name: token_urlsafe(32) for name in ("alice", "bob")}
token_users = {token: name for name, token in tokens.items()}
auth_headers = {
    name: {"Authorization": f"Bearer {token}"} for name, token in tokens.items()
}
bearer = HTTPBearer(auto_error=False)


def get_user(
    credential: Annotated[HTTPAuthorizationCredentials | None, Security(bearer)],
) -> str:
    user_id = token_users.get(credential.credentials) if credential else None
    if user_id is None:
        raise HTTPException(
            401, "需要有效身份", headers={"WWW-Authenticate": "Bearer"}
        )
    return user_id


CurrentUser = Annotated[str, Security(get_user)]

## 5 统一资源归属与提交失败的处理

find_owned 先按编号查记录，再比较当前用户与服务端保存的 owner_id。每个读取、修改和删除操作都要经过它。记录缺失返回 404，记录存在但属于别人返回 403；本例保留这一区别，因此不会隐藏记录是否存在。

In [8]:
def find_owned(session: Session, record_id: int, user_id: str) -> Record:
    record = session.get(Record, record_id)
    if record is None:
        raise HTTPException(404, "记录不存在")
    if record.owner_id != user_id:
        raise HTTPException(403, "无权操作这条记录")
    return record

提交可能因数据库完整性约束失败。本例把 IntegrityError 转成 409“记录约束冲突”，先 rollback 再抛出 HTTPException，避免把数据库异常细节返回给客户端。

这里只演示同一用户标题重复这一约束。增加其他数据库规则后，需要按真实约束区分错误；其他异常不应一律伪装成标题重复。commit 成功后才返回成功响应，不能把提交推迟到响应发出以后。

In [9]:
from sqlalchemy.exc import IntegrityError


def commit_changes(session: Session) -> None:
    try:
        session.commit()
    except IntegrityError as exc:
        session.rollback()
        raise HTTPException(409, "记录约束冲突") from exc

## 6 创建记录，再通过新请求读取

创建时，owner_id 从身份依赖取得，客户端只能提交 title。提交并 refresh 后，把数据库对象转成 RecordOutput，明确限制输出字段。读取也通过同一个输出模型。

装饰器中的 responses 只声明这些路由可能主动返回的错误；请求校验的 422 由 FastAPI 按参数和模型自动补充。

In [10]:
@app.post(
    "/records", status_code=201, response_model=RecordOutput,
    responses={401: errors[401], 409: errors[409]},
)
def create_record(body: RecordInput, user: CurrentUser, session: SessionDep):
    record = Record(title=body.title, owner_id=user)
    session.add(record)
    commit_changes(session)
    session.refresh(record)
    return RecordOutput.model_validate(record)


@app.get(
    "/records/{record_id}", response_model=RecordOutput,
    responses={401: errors[401], 403: errors[403], 404: errors[404]},
)
def read_record(record_id: int, user: CurrentUser, session: SessionDep):
    record = find_owned(session, record_id, user)
    return RecordOutput.model_validate(record)

固定流程从 Alice 创建一条记录开始。随后读取接口响应，再通过一个独立 Session 检查数据库，确认 owner_id 已保存且没有出现在响应中。

In [11]:
with TestClient(app) as client:
    created = client.post(
        "/records", headers=auth_headers["alice"], json={"title": "练习 API"}
    )
    assert created.status_code == 201
    record_id = created.json()["id"]
    record_path = f"/records/{record_id}"
    fetched = client.get(record_path, headers=auth_headers["alice"])
    assert fetched.status_code == 200
    assert fetched.json() == {"id": record_id, "title": "练习 API"}
    with Session(app.state.engine) as session:
        saved = session.get(Record, record_id)
        assert saved.owner_id == "alice" and saved.title == "练习 API"
    print("创建：", created.status_code, created.json())  # 预期：创建： 201 {'id': 1, 'title': '练习 API'}。
    print("查询：", fetched.status_code, fetched.json())  # 预期：查询： 200 {'id': 1, 'title': '练习 API'}。
assert app.state.engine is None

创建： 201 {'id': 1, 'title': '练习 API'}
查询： 200 {'id': 1, 'title': '练习 API'}


## 7 修改和删除复用同一权限边界

PATCH 只修改标题；DELETE 删除整条记录并返回没有响应体的 204。两种操作都先调用 find_owned，再写入并提交，避免读接口有权限检查、写接口却漏掉检查。

In [12]:
@app.patch(
    "/records/{record_id}", response_model=RecordOutput,
    responses={401: errors[401], 403: errors[403], 404: errors[404], 409: errors[409]},
)
def rename_record(
    record_id: int, body: RecordInput, user: CurrentUser, session: SessionDep,
):
    # 更新前先按相同依赖取得身份和会话，再确认记录归属。
    record = find_owned(session, record_id, user)
    record.title = body.title
    commit_changes(session)
    # 提交后刷新对象，再用公开输出模型组织响应。
    session.refresh(record)
    return RecordOutput.model_validate(record)


@app.delete(
    "/records/{record_id}", status_code=204,
    responses={401: errors[401], 403: errors[403], 404: errors[404]},
)
def delete_record(record_id: int, user: CurrentUser, session: SessionDep) -> Response:
    record = find_owned(session, record_id, user)
    # 删除仍复用归属检查；成功返回 204，不附加 JSON 响应体。
    session.delete(record)
    commit_changes(session)
    return Response(status_code=204)

继续固定流程：Alice 修改自己的记录。下面重新进入 TestClient，lifespan 创建新的 Engine，读取的仍是数据库文件里的已提交记录；检查不依赖旧会话中的对象。

In [13]:
with TestClient(app) as client:
    changed = client.patch(
        record_path, headers=auth_headers["alice"], json={"title": "完成 API"}
    )
    assert changed.status_code == 200
    assert changed.json() == {"id": record_id, "title": "完成 API"}
    with Session(app.state.engine) as session:
        saved = session.get(Record, record_id)
        assert saved.title == "完成 API" and saved.owner_id == "alice"
    print("修改：", changed.status_code, changed.json())  # 预期：修改： 200 {'id': 1, 'title': '完成 API'}。

修改： 200 {'id': 1, 'title': '完成 API'}


## 8 拒绝越权和非法输入后，核对数据未变

Bob 是有效用户，仍不能操作 Alice 的记录。依次尝试读取、修改和删除，三个入口都必须拒绝；再用数据库查询确认标题和所有者没有改变。

In [14]:
with TestClient(app) as client:
    denied_statuses = []
    for method in ("GET", "PATCH", "DELETE"):
        arguments = {"json": {"title": "不应写入"}} if method == "PATCH" else {}
        result = client.request(
            method, record_path, headers=auth_headers["bob"], **arguments
        )
        denied_statuses.append(result.status_code)
        assert result.status_code == 403
    with Session(app.state.engine) as session:
        rows = session.exec(select(Record)).all()
        assert len(rows) == 1
        assert (rows[0].title, rows[0].owner_id) == ("完成 API", "alice")
    print("Bob 读取/修改/删除：", denied_statuses)  # 预期：Bob 读取/修改/删除： [403, 403, 403]。
    print("拒绝后标题和所有者未改变")  # 预期：拒绝后标题和所有者未改变。

Bob 读取/修改/删除： [403, 403, 403]
拒绝后标题和所有者未改变


没有通过身份认证时返回 401；已经识别身份但输入不合法时返回 422。下面的无效请求分别测试空标题、伪造 owner_id 和非法编号，只改变一个失败条件，避免把多种错误混在一次请求里。

In [15]:
with TestClient(app) as client:
    missing_auth = client.get(record_path)
    assert missing_auth.status_code == 401
    assert missing_auth.headers["www-authenticate"] == "Bearer"
    for payload in ({"title": ""}, {"title": "伪造", "owner_id": "bob"}):
        invalid = client.patch(record_path, headers=auth_headers["alice"], json=payload)
        assert invalid.status_code == 422
    bad_id = client.get("/records/not-an-integer", headers=auth_headers["alice"])
    assert bad_id.status_code == 422
    unchanged = client.get(record_path, headers=auth_headers["alice"])
    assert unchanged.json() == {"id": record_id, "title": "完成 API"}
    print("无身份/非法输入/非法编号：", missing_auth.status_code,
          invalid.status_code, bad_id.status_code)  # 预期：无身份/非法输入/非法编号： 401 422 422。

无身份/非法输入/非法编号：

 401 422 422


## 9 约束失败先回滚，再完成删除

Alice 再创建一个同名标题会触发联合唯一约束，提交失败后返回 409。数据库中应仍只有原来的记录；随后可以继续发正常请求。

用 Bob 创建相同标题则允许成功，用来确认规则是“同一用户的标题不能重复”，而不是所有用户共享一份标题唯一性。

In [16]:
with TestClient(app) as client:
    duplicate = client.post(
        "/records", headers=auth_headers["alice"], json={"title": "完成 API"}
    )
    assert duplicate.status_code == 409
    assert duplicate.json() == {"detail": "记录约束冲突"}
    with Session(app.state.engine) as session:
        rows = session.exec(select(Record)).all()
        assert len(rows) == 1 and rows[0].id == record_id
        assert (rows[0].title, rows[0].owner_id) == ("完成 API", "alice")
    bob_record = client.post(
        "/records", headers=auth_headers["bob"], json={"title": "完成 API"}
    )
    assert bob_record.status_code == 201
    bob_path = f"/records/{bob_record.json()['id']}"
    print("Alice 重复标题：", duplicate.status_code, duplicate.json())  # 预期：Alice 重复标题： 409 {'detail': '记录约束冲突'}。
    print("Bob 使用相同标题：", bob_record.status_code)  # 预期：Bob 使用相同标题： 201。

Alice 重复标题：

 409 {'detail': '记录约束冲突'}
Bob 使用相同标题： 201


固定流程最后由各自的所有者删除记录。删除后既检查 GET 的 404，也检查数据库为空，不能只依赖一个 204 状态码判断删除成功。

In [17]:
with TestClient(app) as client:
    for name, path in (("alice", record_path), ("bob", bob_path)):
        deleted = client.delete(path, headers=auth_headers[name])
        assert deleted.status_code == 204 and deleted.content == b""
        missing = client.get(path, headers=auth_headers[name])
        assert missing.status_code == 404
        assert missing.json() == {"detail": "记录不存在"}
        print(name, "删除/再次读取：", deleted.status_code, missing.status_code)  # 预期：alice、bob 均为 204 404。
    with Session(app.state.engine) as session:
        remaining = session.exec(select(Record)).all()
        assert remaining == []
    print("数据库记录数：", len(remaining))  # 预期：0，两位用户的记录均已删除。

alice 删除/再次读取： 204 404
bob 删除/再次读取： 204 404
数据库记录数： 0


## 10 把实际行为整理成接口契约

接口契约说明客户端能发送什么、成功返回什么、可能出现哪些错误。下表对应本例实际路由；路径中的 record_id 表示整数记录编号。创建和改名只接受长度为 1–80 的 title，不接受 owner_id。

| 方法与路径 | 成功响应 | 本例声明的错误状态 |
| --- | --- | --- |
| POST /records | 201，含 id 与 title 的 JSON | 401、409、422 |
| GET /records/{record_id} | 200，含 id 与 title 的 JSON | 401、403、404、422 |
| PATCH /records/{record_id} | 200，含 id 与 title 的 JSON | 401、403、404、409、422 |
| DELETE /records/{record_id} | 204，无响应体 | 401、403、404、422 |

主动抛出的 HTTPException 错误体含字符串 detail；默认 422 的 detail 是校验错误列表。下面读取 OpenAPI JSON，核对状态码、输出字段与 Bearer 声明。OpenAPI 能描述这些要求，但权限和事务仍由前面的代码执行。

In [18]:
with TestClient(app) as client:
    schema_response = client.get("/openapi.json")
    assert schema_response.status_code == 200
    schema = schema_response.json()
# 每个路径与方法分别列出可观察到的状态码，包含框架产生的 422。
expected = {
    ("/records", "post"): {"201", "401", "409", "422"},
    ("/records/{record_id}", "get"): {"200", "401", "403", "404", "422"},
    ("/records/{record_id}", "patch"): {"200", "401", "403", "404", "409", "422"},
    ("/records/{record_id}", "delete"): {"204", "401", "403", "404", "422"},
}
for (path, method), codes in expected.items():
    operation = schema["paths"][path][method]
    assert set(operation["responses"]) == codes
    assert operation["security"] == [{"HTTPBearer": []}]
    # 预期：POST 含 201/401/409/422；GET 含 200/401/403/404/422；PATCH 另含 409；DELETE 含 204/401/403/404/422。
    print(method.upper(), path, sorted(operation["responses"]))
# 输出模型与 204 无响应体是两项独立的接口契约。
output_fields = schema["components"]["schemas"]["RecordOutput"]["properties"]
assert set(output_fields) == {"id", "title"}
delete_response = schema["paths"]["/records/{record_id}"]["delete"]["responses"]["204"]
assert "content" not in delete_response
print("输出字段：", list(output_fields))  # 预期：['id', 'title']，不包含 owner_id。

POST /records ['201', '401', '409', '422']
GET /records/{record_id} ['200', '401', '403', '404', '422']
PATCH /records/{record_id} ['200', '401', '403', '404', '409', '422']
DELETE /records/{record_id} ['204', '401', '403', '404', '422']
输出字段： ['id', 'title']


## 11 关闭资源并说明交付边界

运行入口、输入、成功响应、错误行为和退出方式都应能由读者复现。本篇从空内核顺序执行即可完成固定流程，所有 TestClient 都使用 with；因此先退出请求和会话，再由 lifespan 释放 Engine，最后删除临时文件。

dispose 释放连接池中的连接，不会强行关闭仍被会话占用的连接，也不会禁止以后重新连接；本例退出后不再使用旧 Engine。数据库文件保存已提交数据，本篇为了实验隔离主动删除它。

交付范围：这是本地教学 API，验证的是应用内调用。没有真实 HTTP 监听、账号系统或数据库迁移；临时身份映射属于当前进程，不能直接用于多 worker 共享身份。投入实际服务前，应选定身份提供方、数据保存与迁移方案，并另行检查网络部署。

In [19]:
assert app.state.engine is None
starts = lifecycle_events.count("启动")
releases = lifecycle_events.count("释放")
assert starts == releases
print("应用启动/资源释放次数：", starts, releases)  # 预期：应用启动/资源释放次数： 8 8。

database_path = settings.database_path
work_directory.cleanup()
assert not database_path.exists()
assert not database_path.parent.exists()
tokens.clear()
token_users.clear()
auth_headers.clear()
print("临时数据库目录已删除：", not database_path.parent.exists())  # 预期：临时数据库目录已删除： True。
# 此后重新调用接口需要从空内核运行，重新准备数据库和临时身份。

应用启动/资源释放次数： 8 8
临时数据库目录已删除： True


## 本章小结

（1）配置提供数据库位置，lifespan 管理 Engine，依赖为每次请求提供独立 Session。

（2）身份决定当前用户，归属检查决定能否操作目标记录；输入只接受客户端有权修改的字段。

（3）提交成功才返回成功；约束失败先回滚，越权和校验失败后核对数据库未变。

（4）固定请求流程、数据库查询和 OpenAPI 检查互相补充；退出时关闭应用资源并清理临时输入。

## 练习

练习在末节清理前完成；如果已经运行完整篇，先从空内核重新运行到相关示例，结束后执行清理。

（1）让 Alice 创建两个不同标题，再把第二条改成第一条的标题。验证标准：PATCH 返回 409，独立 Session 仍能读到两条原始标题，随后合法改名能成功。

（2）添加“只列出当前用户记录”的 GET /records。验证标准：两个用户分别创建记录后，只能列出自己的 id 和 title，响应中没有 owner_id，并为新接口核对 OpenAPI。

（3）把合法输入中的 title 改为 None、81 个字符或缺失字段。验证标准：每种情况都返回 422，数据库记录数不增加；再发一个合法创建请求仍成功。

（4）给固定流程增加一条不存在记录的 DELETE 请求。验证标准：身份有效时返回 404，现有记录没有被删除，最终所有应用启动都对应资源释放，临时目录也被删除。

提示：错误响应与数据库结果一起断言；列表查询把 owner_id 条件放在服务端。第（1）题检查回滚后的持久化值，不使用失败前保存在 Python 对象里的值作唯一依据。

## 参考与引用来源

- **FastAPI 官方文档**：[Settings and Environment Variables](https://fastapi.tiangolo.com/advanced/settings/#pydantic-settings) 与 [Lifespan](https://fastapi.tiangolo.com/advanced/events/#lifespan)，用于配置和应用资源范围；[Dependencies with yield](https://fastapi.tiangolo.com/tutorial/dependencies/dependencies-with-yield/) 与 [Testing Events](https://fastapi.tiangolo.com/advanced/testing-events/)，用于请求资源和 TestClient 生命周期；[HTTPBearer](https://fastapi.tiangolo.com/reference/security/#fastapi.security.HTTPBearer)、[Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/) 与 [Additional Responses in OpenAPI](https://fastapi.tiangolo.com/advanced/additional-responses/)，用于读取身份凭据、实际错误响应和错误声明。
- **SQLModel 官方文档**：[Multiple Models](https://sqlmodel.tiangolo.com/tutorial/fastapi/multiple-models/)、[Session with FastAPI Dependency](https://sqlmodel.tiangolo.com/tutorial/fastapi/session-with-dependency/) 的 get_session 与路由示例、[Update Data](https://sqlmodel.tiangolo.com/tutorial/fastapi/update/)、[Delete Data](https://sqlmodel.tiangolo.com/tutorial/fastapi/delete/)，用于表模型、请求会话、提交、刷新及增删改查。
- **SQLAlchemy 2.0 官方文档**：[UNIQUE Constraint](https://docs.sqlalchemy.org/en/20/core/constraints.html#unique-constraint) 与同页 Setting up Constraints when using the Declarative ORM Extension，用于联合唯一约束和表级配置；[Session Basics](https://docs.sqlalchemy.org/en/20/orm/session_basics.html) 的 Committing、Rolling Back、Closing 和 Is the Session thread-safe，用于事务失败、回滚和会话范围；[Engine Disposal](https://docs.sqlalchemy.org/en/20/core/connections.html#engine-disposal) 与 [SQLite 的 Threading/Pooling Behavior](https://docs.sqlalchemy.org/en/20/dialects/sqlite.html#threading-pooling-behavior)，用于释放连接及同步 SQLite 的线程边界；[Altering Database Objects through Migrations](https://docs.sqlalchemy.org/en/20/core/metadata.html#altering-database-objects-through-migrations)，用于建表与迁移的区别。
- **Pydantic 官方文档**：[Settings Management](https://pydantic.dev/docs/validation/latest/concepts/pydantic_settings/) 的 Usage 与 Environment variable names、[Models](https://pydantic.dev/docs/validation/latest/concepts/models/) 的 Extra data 与 Arbitrary class instances、[Fields](https://pydantic.dev/docs/validation/latest/concepts/fields/) 的 Field constraints，用于配置输入、额外字段、属性转换和标题长度约束。
- **OWASP Cheat Sheet Series**：[Authorization Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Authorization_Cheat_Sheet.html) 的 Validate the Permissions on Every Request、Enforce Authorization Checks on the Right Location 和 Create Unit and Integration Test Cases for Authorization Logic，用于服务端逐请求授权及拒绝后的数据检查。
- **RFC Editor**：[RFC 9110 第 15 节](https://www.rfc-editor.org/rfc/rfc9110.html#section-15)，用于 201、204、401、403、404、409 和 422 的语义。
- **Python 官方文档**：[secrets.token_urlsafe](https://docs.python.org/3/library/secrets.html#secrets.token_urlsafe) 与 [TemporaryDirectory](https://docs.python.org/3/library/tempfile.html#tempfile.TemporaryDirectory)，用于运行时随机凭据和临时目录清理。